# Ingest Latest Videos for Business Cluster

This notebook fetches the latest video metadata and statistics from the YouTube API for the channels in the Graphiko business cluster, calculates engagement trends, and generates title embeddings for downstream semantic analysis.

## Environment Setup and Authentication

This cell installs the necessary dependencies (YouTube API client, Pinecone, pandas, etc.) and establishes connections to Google Drive and external services. It uses Colab `userdata` for secrets like API keys.

In [1]:
# Install dependencies
!pip install -q google-api-python-client pinecone pandas numpy

import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
try:
    from google.colab import drive, userdata
except ImportError:
    drive, userdata = None, None
from googleapiclient.discovery import build
from pinecone import Pinecone
from pathlib import Path

# Mount Google Drive
try:
    if drive:
        drive.mount('/content/drive')
        print("✅ Drive mounted")
except Exception as e:
    print(f"⚠️ Drive mount failed (local execution?): {e}")

# Initialize YouTube API
try:
    YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY') if userdata else None
    if YOUTUBE_API_KEY:
        youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
        print("✅ YouTube API initialized")
    else:
        print("⚠️ No YouTube API key found")
except Exception as e:
    print(f"❌ YouTube API setup failed: {e}")

# Initialize Pinecone
try:
    PINECONE_API_KEY = userdata.get('PINECONE_API_KEY') if userdata else None
    if PINECONE_API_KEY:
        pc = Pinecone(api_key=PINECONE_API_KEY)
        pinecone_index = pc.Index('finder')
        print("✅ Pinecone connected")
    else:
        print("⚠️ No Pinecone API key found")
except Exception as e:
    print(f"❌ Pinecone setup failed: {e}")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


⚠️ No YouTube API key found
⚠️ No Pinecone API key found


## Load Unique Channels from 20D Data

We load the canonical 20D video embeddings artifact to identify the specific channels belonging to the business cluster. This ensures our ingestion remains aligned with the established research universe.

In [2]:
DATA_20D_PATH = '/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv'

try:
    df_20d = pd.read_csv(DATA_20D_PATH)
    unique_channels = df_20d[['channel_id', 'channel_name']].drop_duplicates().to_dict('records')
    print(f"Loaded {len(unique_channels)} unique channels from business cluster.")
    for ch in unique_channels[:5]:
        print(f" - {ch['channel_name']} ({ch['channel_id']})")
except Exception:
    print(f"❌ 20D data not found at {DATA_20D_PATH}. Loading empty channel list.")
    unique_channels = []

❌ 20D data not found at /content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv. Loading empty channel list.


## Fetch Last 50 Videos from YouTube API

For each identified channel, we fetch the latest 50 videos using the YouTube Search and Videos endpoints to retrieve snippets (titles, dates) and statistics (view, like, and comment counts).

In [3]:
def fetch_latest_videos(channel_id, max_results=50):
    if 'youtube' not in globals() or youtube is None: return []
    # 1. Search for latest video IDs
    search_response = youtube.search().list(
        channelId=channel_id,
        part='id,snippet',
        order='date',
        type='video',
        maxResults=max_results
    ).execute()

    video_ids = [item['id']['videoId'] for item in search_response.get('items', []) if 'videoId' in item['id']]
    if not video_ids:
        return []

    # 2. Get full statistics for these videos
    stats_response = youtube.videos().list(
        part='snippet,statistics',
        id=','.join(video_ids)
    ).execute()

    videos = []
    for item in stats_response.get('items', []):
        snippet = item['snippet']
        stats = item['statistics']
        videos.append({
            'video_id': item['id'],
            'channel_id': channel_id,
            'title': snippet['title'],
            'published_at': snippet['publishedAt'],
            'view_count': int(stats.get('viewCount', 0)),
            'like_count': int(stats.get('likeCount', 0)),
            'comment_count': int(stats.get('commentCount', 0))
        })
    return videos

all_new_videos = []
for channel in unique_channels:
    try:
        print(f"Fetching videos for: {channel['channel_name']}...")
        vids = fetch_latest_videos(channel['channel_id'])
        all_new_videos.extend(vids)
    except Exception as e:
        print(f" ⚠️ Failed to fetch {channel['channel_name']}: {e}")

df_ingested = pd.DataFrame(all_new_videos)
if not df_ingested.empty:
    df_ingested['published_at'] = pd.to_datetime(df_ingested['published_at'])
print(f"Total ingested videos: {len(df_ingested)}")

Total ingested videos: 0


## Descriptive Statistics and Performance Trends

This section calculates the average channel performance and compares the metrics from the last month against the previous year to identify growth or decline trends. Note: "Yearly" stats are limited to the fetched sample (last 50 videos).

In [4]:
if not df_ingested.empty:
    # Ensure we are comparing apples to apples with timezone-aware datetimes
    now = datetime.now(timezone.utc)
    last_month = now - timedelta(days=30)
    last_year = now - timedelta(days=365)

    stats_list = []
    for channel_id, group in df_ingested.groupby('channel_id'):
        # Handle timezone awareness for comparison
        month_vids = group[group['published_at'] > last_month]
        year_vids = group[group['published_at'] > last_year]
        
        month_avg = month_vids['view_count'].mean() if not month_vids.empty else 0
        year_avg = year_vids['view_count'].mean() if not year_vids.empty else 0
        
        trend = (month_avg / year_avg) - 1 if year_avg > 0 else 0
        
        stats_list.append({
            'channel_id': channel_id,
            'avg_views_sample': group['view_count'].mean(),
            'month_avg_views': month_avg,
            'year_avg_views': year_avg, # Based on sample
            'trend': trend
        })

    df_stats = pd.DataFrame(stats_list)
    print("Descriptive statistics calculated.")
else:
    df_stats = pd.DataFrame(columns=['channel_id', 'avg_views_sample', 'month_avg_views', 'year_avg_views', 'trend'])
    print("No videos ingested, skipping stats.")
df_stats.head()

No videos ingested, skipping stats.


,channel_id,avg_views_sample,month_avg_views,year_avg_views,trend


## Video Title Embedding (Pinecone)

We embed the ingested video titles using Pinecone's inference API. We implement a fetch-or-embed pattern to avoid redundant computations for videos already present in the `VideoTitles` namespace.

In [5]:
def fetch_or_embed_titles(index, titles_dict, namespace='VideoTitles', model="multilingual-e5-large"):
    if 'pc' not in globals() or pc is None: return {}
    ids = list(titles_dict.keys())
    embeddings = {}
    
    # Batch fetch existing
    for i in range(0, len(ids), 100):
        batch_ids = ids[i:i+100]
        res = index.fetch(ids=batch_ids, namespace=namespace)
        for vid, data in res.get('vectors', {}).items():
            embeddings[vid] = data['values']
            
    missing_ids = [vid for vid in ids if vid not in embeddings]
    print(f"Found {len(embeddings)} existing embeddings, need to generate {len(missing_ids)}.")
    
    # Embed missing
    for i in range(0, len(missing_ids), 96):
        batch_ids = missing_ids[i:i+96]
        batch_texts = [titles_dict[vid] for vid in batch_ids]
        
        res = pc.inference.embed(model=model, inputs=batch_texts, parameters={"input_type": "passage"})
        
        to_upsert = []
        for vid, emb in zip(batch_ids, res.data):
            embeddings[vid] = emb['values']
            to_upsert.append({
                'id': vid,
                'values': emb['values'],
                'metadata': {'title': titles_dict[vid]}
            })
        
        index.upsert(vectors=to_upsert, namespace=namespace)
        
    return embeddings

if not df_ingested.empty:
    titles_dict = df_ingested.set_index('video_id')['title'].to_dict()
    all_embeddings = fetch_or_embed_titles(pinecone_index, titles_dict)
    print("Embedding process complete.")
else:
    all_embeddings = {}
    print("No videos ingested, skipping embeddings.")

No videos ingested, skipping embeddings.


## Versioned Data Export

Finally, we merge the metadata, statistics, and embeddings into a unified dataset and save it to Google Drive in a timestamped, versioned directory for use in subsequent analysis steps.

In [6]:
if not df_ingested.empty:
    version = datetime.now().strftime('%Y%m%d_%H%M%S')
    # Use current directory if Drive is not mounted
    base_dir = Path('/content/drive/MyDrive/Graphiko/ingesto') if drive else Path('./ingesto')
    export_path = base_dir / version
    export_path.mkdir(parents=True, exist_ok=True)
    latest_path = base_dir / 'latest'
    latest_path.mkdir(parents=True, exist_ok=True)

    # Attach embeddings to dataframe
    df_ingested['embedding'] = df_ingested['video_id'].map(all_embeddings)

    # Merge stats
    df_final = df_ingested.merge(df_stats, on='channel_id', how='left')

    # Save
    df_final.to_pickle(export_path / 'ingested_videos.pkl')
    df_final.to_csv(export_path / 'ingested_videos.csv', index=False)
    df_final.to_csv(latest_path / 'ingested_videos.csv', index=False)

    print(f"✅ Exported version {version}.")
else:
    print("No data to export.")

No data to export.
